# Day 6: FastAPI Core and SQLAlchemy Intro
**Routing, parameters, Pydantic models, full CRUD and ORM concepts**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi uvicorn httpx sqlalchemy pydantic

# ===== Option B: VS Code / local (run in your activated venv terminal) =====
# pip install fastapi uvicorn httpx sqlalchemy pydantic
import fastapi, pydantic, sqlalchemy
print(fastapi.__version__, pydantic.__version__, sqlalchemy.__version__)

## 2. Pydantic on its own

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BookCreate(BaseModel):
    title: str = Field(min_length=1, examples=["Ikigai"])
    author: str
    pages: int = Field(gt=0)
    year: int = Field(ge=1450, le=2026)
    tags: list[str] = []

ok = BookCreate(title="Ikigai", author="Garcia", pages=208, year=2016)
print(ok)
print(ok.model_dump())

try:
    BookCreate(title="", author="X", pages=-5, year=3000)
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], "->", err["msg"])

## 3. A Books API with routers, models and full CRUD

In [ ]:
%%writefile books_api.py
from fastapi import APIRouter, FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

class BookCreate(BaseModel):
    title: str = Field(min_length=1)
    author: str
    pages: int = Field(gt=0)

class BookUpdate(BaseModel):
    title: str | None = None
    author: str | None = None
    pages: int | None = Field(default=None, gt=0)

class Book(BookCreate):
    id: int

books: dict[int, Book] = {}
router = APIRouter(prefix="/books", tags=["Books"])

@router.get("/", response_model=list[Book])
def list_books(author: str | None = None, limit: int = Query(10, ge=1, le=100)):
    result = [b for b in books.values() if author is None or b.author == author]
    return result[:limit]

@router.get("/{book_id}", response_model=Book)
def get_book(book_id: int):
    if book_id not in books:
        raise HTTPException(404, "Book not found")
    return books[book_id]

@router.post("/", response_model=Book, status_code=201)
def create_book(data: BookCreate):
    new_id = max(books, default=0) + 1
    books[new_id] = Book(id=new_id, **data.model_dump())
    return books[new_id]

@router.patch("/{book_id}", response_model=Book)
def update_book(book_id: int, data: BookUpdate):
    if book_id not in books:
        raise HTTPException(404, "Book not found")
    changes = data.model_dump(exclude_unset=True)
    books[book_id] = books[book_id].model_copy(update=changes)
    return books[book_id]

@router.delete("/{book_id}", status_code=204)
def delete_book(book_id: int):
    if books.pop(book_id, None) is None:
        raise HTTPException(404, "Book not found")

app = FastAPI(title="Books API")
app.include_router(router)

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, books_api
importlib.reload(books_api)
from fastapi.testclient import TestClient
client = TestClient(books_api.app)

# ===== Option B: VS Code =====
# Run in the terminal:  uvicorn books_api:app --reload
# Then open http://127.0.0.1:8000/docs and try each endpoint there.

r = client.post("/books/", json={"title": "Ikigai", "author": "Garcia", "pages": 208})
print(r.status_code, r.json())
client.post("/books/", json={"title": "Deep Work", "author": "Newport", "pages": 296})
print(client.get("/books/", params={"limit": 5}).json())

In [ ]:
print("PATCH:", client.patch("/books/1", json={"pages": 210}).json())
print("Bad data:", client.post("/books/", json={"title": "", "author": "X", "pages": 0}).status_code)
print("Missing:", client.get("/books/99").status_code)
print("Delete:", client.delete("/books/2").status_code)
print("Left:", [b["title"] for b in client.get("/books/").json()])

### TODO 1: Add a `year` field (1450 to 2026) to the models and a query filter `min_pages` to the list endpoint.

In [ ]:
# TODO: write your code here

## 4. ORM basics with SQLAlchemy
SQLite is a database stored in a single file. No server needed, so it works in Colab and VS Code alike.

In [ ]:
import os
from sqlalchemy import create_engine, select, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

if os.path.exists("library.db"):
    os.remove("library.db")
engine = create_engine("sqlite:///library.db", echo=False)   # echo=True prints the SQL

class Base(DeclarativeBase):
    pass

class BookRow(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    author: Mapped[str]
    pages: Mapped[int]

Base.metadata.create_all(engine)
print("Tables:", list(Base.metadata.tables))

In [ ]:
with Session(engine) as s:
    s.add_all([
        BookRow(title="Ikigai", author="Garcia", pages=208),
        BookRow(title="Deep Work", author="Newport", pages=296),
        BookRow(title="Atomic Habits", author="Clear", pages=320),
        BookRow(title="The Little Prince", author="Saint-Exupery", pages=96),
    ])
    s.commit()

with Session(engine) as s:
    stmt = select(BookRow).where(BookRow.pages > 200).order_by(BookRow.title)
    for b in s.scalars(stmt):
        print(b.id, b.title, b.pages)

In [ ]:
with Session(engine) as s:
    book = s.get(BookRow, 1)
    book.pages = 210                  # update: change the attribute
    s.delete(s.get(BookRow, 4))       # delete
    s.commit()
    print([(b.title, b.pages) for b in s.scalars(select(BookRow))])

## 5. ORM object to Pydantic response

In [ ]:
class BookOut(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    title: str
    pages: int

with Session(engine) as s:
    rows = s.scalars(select(BookRow)).all()
    print([BookOut.model_validate(r).model_dump() for r in rows])

### TODO 2: Turn on `echo=True` in `create_engine` and rerun a query. Read the SQL that SQLAlchemy writes for you.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)